In [ ]:
#!/usr/bin/env python3
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ---------- Load ----------
ds = xr.open_dataset('/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/Processed_Data/filtered_mcstrack_data_time_period_new.nc')
t_index = 0
tb            = ds['tb'].isel(time=t_index)
precip        = ds['precipitation'].isel(time=t_index).where(lambda x: x >= 0)
cloudtracknum = ds['cloudtracknumber'].isel(time=t_index)
cloudtype     = ds['cloudtype'].isel(time=t_index)
lat, lon      = ds['lat'], ds['lon']

# ---------- Colormaps / norms ----------
tb_cmap = plt.get_cmap("turbo")

eps = 0.1  # values < eps plot as white
precip_cmap = plt.get_cmap("Spectral_r").copy()
precip_cmap.set_under("white")
precip_norm = mcolors.Normalize(vmin=eps, vmax=10.0)

# cloudtype: 1=core, 2=cold anvil, 3=warm anvil
cloudtype_cmap = ListedColormap(["#0000FF", "#FF0000", "green"])
cloudtype_cmap.set_bad("white")
cloudtype_norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5], cloudtype_cmap.N)  # 1,2,3

track_cmap = plt.get_cmap("gnuplot")

# ---------- Figure & GridSpec (no middle gap) ----------
fig = plt.figure(figsize=(22, 14), constrained_layout=False)
gs  = fig.add_gridspec(
    2, 2,
    left=0.04, right=0.80, bottom=0.06, top=0.96,
    wspace=0.1, hspace=0.35
)

ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
ax10 = fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree())
ax11 = fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree())
axs  = [ax00, ax01, ax10, ax11]

# Force axes to fill the allocated boxes (removes the center gap)
for i, ax in enumerate(axs):
    ax.set_aspect("auto")
    ax.set_anchor("W" if i % 2 == 0 else "E")  # left col sticks to W, right col to E

extent = [55, 115, -5, 40]  # AMA region

def add_gridlabels(ax, is_left, is_bottom):
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = is_left
    gl.bottom_labels = is_bottom
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 25, 'rotation': 35}
    gl.ylabel_style = {'size': 25, 'rotation': 35}

def panel(ax, data, title, cmap, units="",
          vmin=None, vmax=None, norm=None,
          is_bottom=False, discrete_ticks=None,
          ticklabels=None):
    """
    Generic map panel with inset horizontal colorbar.
    If discrete_ticks is given, you can optionally pass ticklabels
    to override the default numeric labels.
    """
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    im = ax.pcolormesh(
        lon, lat, data,
        cmap=cmap, vmin=vmin, vmax=vmax, norm=norm,
        transform=ccrs.PlateCarree(),
        shading="auto", rasterized=True
    )
    ax.coastlines(linewidth=2)
    ax.add_feature(cfeature.BORDERS, linewidth=2)
    ax.set_title(title, fontsize=30, pad=8)

    # Inset colorbar (does not resize axes)
    y_off = -0.25 if is_bottom else -0.10
    cax = inset_axes(
        ax, width="70%", height="6%", loc="lower center",
        bbox_to_anchor=(0.0, y_off, 1.0, 1.0),
        bbox_transform=ax.transAxes, borderpad=0
    )
    cb = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.ax.tick_params(labelsize=25)
    if units:
        cb.set_label(units, fontsize=25)

    if discrete_ticks is not None:
        cb.set_ticks(discrete_ticks)
        if ticklabels is None:
            cb.set_ticklabels([str(t) for t in discrete_ticks])
        else:
            cb.set_ticklabels(ticklabels)
    else:
        cb.locator = mticker.MaxNLocator(nbins=5)
        cb.update_ticks()

    return im

# ---- Panels ----
panel(ax00, tb, r"$\bf{(a)}$ Brightness temperature", tb_cmap,
      "K", vmin=200, vmax=300, is_bottom=False)

panel(ax01, precip, r"$\bf{(b)}$ Precipitation", precip_cmap,
      "mm h$^{-1}$", norm=precip_norm, is_bottom=False)

# Subplot (c): Cloud type with text labels for categories
panel(
    ax10, cloudtype, r"$\bf{(c)}$ Cloud type", cloudtype_cmap,
    "Cloud type", norm=cloudtype_norm, is_bottom=True,
    discrete_ticks=[1, 2, 3],
    ticklabels=["core", "cold", "warm"]
)

panel(
    ax11, cloudtracknum, r"$\bf{(d)}$ Cloud track number", track_cmap,
    "unitless",
    vmin=float(cloudtracknum.min()), vmax=float(cloudtracknum.max()),
    is_bottom=True
)

# Grid labels
add_gridlabels(ax00, True,  False)
add_gridlabels(ax01, False, False)
add_gridlabels(ax10, True,  True)
add_gridlabels(ax11, False, True)

# Save figure
fig.savefig(
    '/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/Figure1_MCS_Pixel_level_snapshot.png',
    dpi=150,
    bbox_inches="tight"
)
fig.savefig(
    '/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/Figure1_MCS_Pixel_level_snapshot.pdf',
    dpi=150,
    bbox_inches="tight"
)
# plt.show()  # optional if you want to see it interactively
